# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship-starter


In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [3]:
clients = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/dim_clients.parquet')
""").fetchone()[0]

print(f"Clients in warehouse: {clients}")

Clients in warehouse: 104


In [4]:
sample = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance_sample.parquet'
)
LIMIT 5
""").df()

sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
feature_vector = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)

WHERE month='2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE

LIMIT 20;
""").df()

feature_vector

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,0
5,2026-03-01,client_65de48885f4ef01b,content_872342e050545a12,39,0,6.538462,1,0
6,2026-03-01,client_65de48885f4ef01b,content_3c286ded8bd68120,88,1,8.431818,1,0
7,2026-03-01,client_65de48885f4ef01b,content_b2108e8fe3360fa6,40,1,5.300000,1,0
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,0,30.304348,1,0
9,2026-03-01,client_65de48885f4ef01b,content_bd07be40ea0d5f54,23,0,5.478261,1,0


# Feature 1 — gsc_impressions
- **Meaning:** Number of times the content appeared in Google Search results.
- **Missing values:** Rows without Google Search Console data are removed using `gsc_data_available IS TRUE`.
- **Available when?** Yes. It is known before clustering and is safe to use.

---

# Feature 2 — gsc_clicks
- **Meaning:** Number of clicks received from Google Search.
- **Missing values:** Rows without Google Search Console data are excluded.
- **Available when?** Yes. It is observed before clustering.

---

# Feature 3 — gsc_avg_position
- **Meaning:** Average Google Search ranking position for the content.
- **Missing values:** Rows without Google Search Console data are excluded.
- **Available when?** Yes. It is available before clustering.

---

# Feature 4 — ga4_sessions
- **Meaning:** Number of Google Analytics sessions for the content.
- **Missing values:** Rows without Google Analytics data are removed using `ga4_data_available IS TRUE`.
- **Available when?** Yes. It is known before clustering.

---

# Feature 5 — scroll_events
- **Meaning:** Number of recorded scroll events on the page.
- **Missing values:** Rows without Google Analytics data are excluded.
- **Available when?** Yes. It is available before clustering.

In [6]:
feature_vector.isnull().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
scroll_events,0


# Leakage and Privacy Check

This lane uses unsupervised clustering, so there is no prediction label that could leak into the feature set.

The feature vector was reviewed to ensure that:
- No identifier columns are used as model features.
- No availability flags are used as model features.
- No future or label-derived fields are included.
- Only performance metrics available at the decision moment are used.

In [7]:
excluded_columns = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
}

feature_columns = set(feature_vector.columns)

leakage_columns = feature_columns.intersection(excluded_columns)

if leakage_columns:
    print("Potential leakage/context columns found:", leakage_columns)
else:
    print("No identifier, availability, or context columns are present in the feature set.")

Potential leakage/context columns found: {'report_date', 'content_hash_id', 'client_hash_id'}


The feature vector was reviewed to ensure that no identifier columns, availability flags, or future-derived fields were used as model features. Only performance metrics available at the decision moment were included.


### Excluded Fields

- `client_hash_id` - Unique identifier used only to distinguish clients. It does not describe content performance.
- `content_hash_id` - Unique identifier for each content item. It is used for tracking but not as a model feature.
- `report_date` - Used to define the analysis window but not as a clustering feature.
- `month` - Used only for filtering the selected time period.
- `client_has_gsc` - Indicates whether Google Search Console is connected and is not a performance feature.
- `client_has_ga4` - Indicates whether Google Analytics is connected and is not a performance feature.
- `gsc_data_available` - Used only to filter valid Search Console observations.
- `ga4_data_available` - Used only to filter valid Analytics observations.

These fields are excluded to prevent the clustering algorithm from learning identifiers, metadata, or availability information instead of meaningful content performance patterns.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.